<a href="https://colab.research.google.com/github/Gael199/Final_Projet/blob/master/02_Exe_MapReduce_Eudes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h3>Preparing the Environment</h3>

<p>
Before we can explore Hadoop and MapReduce inside Google Colab, we first need to set up the foundations that will allow the Hadoop ecosystem to run smoothly in this cloud environment.
Since Hadoop relies on Java, we begin by making sure the correct Java version is installed.
Once the Java runtime is ready, we bring in a fresh copy of Hadoop from the official Apache repository and unpack it directly into the Colab workspace.
This step essentially prepares our local “mini cluster,” giving us all the tools we will use throughout the rest of this notebook as we move toward executing real MapReduce jobs on actual data.
</p>

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jre-headless_8u462-ga%7eus1-0ubuntu2%7e22.04.2_amd64.deb  404  Not Found [IP: 91.189.92.24 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jdk-headless_8u462-ga%7eus1-0ubuntu2%7e22.04.2_amd64.deb  404  Not Found [IP: 91.189.92.24 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
!apt-get update -y
!apt-get install -y openjdk-11-jdk

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,510 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

<h3>Configuring the Hadoop Environment</h3>

<p>
With Hadoop downloaded, the next step is to make Colab aware of where everything lives.
In this part of the setup, we define a few essential environment variables that allow the rest of the notebook to interact with Hadoop just as it would on a regular cluster.
We tell the system where Java is installed, point it to the location of our Hadoop folder, and finally extend the system path so that Hadoop commands can be executed naturally from any cell.
After this configuration, our Colab runtime behaves like a lightweight Hadoop environment, ready for the tasks that follow.
</p>

In [ ]:
# Configuration du cours qui ne fonctione pas
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += f":{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin"

In [3]:
# Ma propre configuration
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += f":{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin"

<h3>Setting Up Hadoop’s Core Configuration</h3>

<p>
Now that the environment is prepared, we customize Hadoop’s core configuration to match the way we intend to use it inside Colab.
Since we are not running a distributed cluster here, we configure Hadoop to operate in local mode by telling it to use the local filesystem instead of HDFS.
This small adjustment ensures that Hadoop works directly with the files stored in our Colab workspace, allowing us to run MapReduce jobs smoothly without needing a full cluster.
It is a lightweight setup, but perfectly suited for demonstrations, teaching, and hands-on experimentation.
</p>

In [4]:
%%bash
cat > /content/hadoop-3.3.6/etc/hadoop/core-site.xml << EOF
<configuration>
 <property>
   <name>fs.defaultFS</name>
   <value>file:///</value>
 </property>
</configuration>
EOF

<h2>MapReduce Mini-Project: Analyzing Amazon Movie Reviews</h2>

<p>
In this exercise, you will work as a data engineer for a streaming platform.
Your goal is to perform several analytics tasks on a free and publicly
available dataset of Amazon Movie Reviews using MapReduce in Hadoop.
</p>

<p>
You will complete four tasks:
</p>

<ol>
  <li><b>Count total number of reviews per movie</b></li>
  <li><b>Compute average rating per movie</b></li>
  <li><b>Extract frequent keywords from reviews</b></li>
  <li><b>Join average ratings with top keywords</b></li>
</ol>

<p>
For each task, you will write a MapReduce program (Python Streaming or Java)
and run it using Hadoop in local mode. Your final outputs will help the
company understand which movies are popular, how viewers rate them, and what
keywords often appear in the reviews.
</p>

<h2>About the Dataset</h2>

<p>
We will use the <b>Amazon Movies &amp; TV 5-core dataset</b>, which is publicly
available and contains movie reviews from Amazon. Each entry in the dataset
is stored as a JSON object with fields such as:
</p>

<ul>
  <li><code>reviewerID</code> – the ID of the reviewer</li>
  <li><code>asin</code> – unique movie identifier</li>
  <li><code>reviewText</code> – full written review</li>
  <li><code>overall</code> – the star rating (1 to 5)</li>
  <li><code>vote</code> – how many users found the review helpful</li>
  <li><code>category</code> – always “Movies &amp; TV” in this dataset</li>
</ul>

<p>
You will download the dataset and inspect a few records to understand its
structure before starting the tasks.
</p>

In [13]:
import gzip
import json
import os
import sys # Import sys for printing warnings to stderr

# -------------------------------------------------------------------
# 1) Download the SMALL Movies & TV dataset (correct version)
# -------------------------------------------------------------------
print("Downloading SMALL Movies & TV 5-core dataset...")

URL = "https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz"
FILE_GZ = "Movies_and_TV_small.json.gz"

!wget --no-check-certificate -O {FILE_GZ} {URL}

if os.path.getsize(FILE_GZ) == 0:
    raise ValueError("Downloaded file is empty!")

print("Download complete.\n")

# -------------------------------------------------------------------
# 2) Load JSON data (each line is a JSON object)
# -------------------------------------------------------------------
print("Loading JSON data from JSON Lines format...")

data = []
with gzip.open(FILE_GZ, "rt", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if line: # Only process non-empty lines
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Could not decode JSON on line {line_num}: {line}. Error: {e}", file=sys.stderr)
                # Continue to the next line to be robust against malformed lines
                continue

print(f"Total records loaded: {len(data)}") # Should be ~3.4 million records
print()

# -------------------------------------------------------------------
# 3) Convert to JSON-LINES format for MapReduce (if not already done)
#    This step ensures 'movies.json' is a clean JSON-Lines file.
# -------------------------------------------------------------------
print("Converting to JSON-lines format (outputting to movies.json with 900,000 records)...")

# Limit to 900,000 records to have less runing times on colab (on a real cluster, remove this line)
limited_data = data[:900000]

with open("movies.json", "w", encoding="utf-8") as out:
    for entry in limited_data:
        out.write(json.dumps(entry) + "\n")

print(f"Conversion complete. Saved as movies.json with {len(limited_data)} records\n")

# -------------------------------------------------------------------
# 4) Preview
# -------------------------------------------------------------------
print("Sample entries:\n")

with open("movies.json", "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        if not line: # Check for end of file
            print("Not enough lines in movies.json to display 3 samples.")
            break
        print(json.loads(line))

--2025-12-07 12:39:44--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 791322468 (755M) [application/x-gzip]
Saving to: ‘Movies_and_TV_small.json.gz’

Movies_and_TV_small 100%[===================>] 754.66M  22.8MB/s    in 34s     

2025-12-07 12:40:18 (22.5 MB/s) - ‘Movies_and_TV_small.json.gz’ saved [791322468/791322468]

Download complete.

Loading JSON data from JSON Lines format...
Total records loaded: 3410019

Converting to JSON-lines format (outputting to movies.json with 900,000 records)...
Conversion complete. Saved as movies.json with 900000 records

Sample entries:

{'overall': 5.0, 'verified': True, 'reviewTime': '11 9, 2012', 'reviewerID': 'A2M1CU2IRZG0K9', 'asin': '0005089549', 'st

<h2>Task 1 — Count Total Number of Reviews per Movie</h2>

<p>
Your first task is to count how many reviews each movie has received. You will
write a MapReduce program where:
</p>

<ul>
  <li>The <b>mapper</b> reads each JSON record, extracts the <code>asin</code>
      field, and emits <code>(asin, 1)</code>.</li>
  <li>The <b>reducer</b> sums the counts for each movie and outputs
      <code>(asin, total_reviews)</code>.</li>
</ul>

<p>
This task is conceptually similar to a word count, but applied to movie IDs.
Complete the mapper and reducer code in the following cell.
</p>

In [ ]:
# Write your Mapper and Reducer code for Task 1 here.
# You may use Python Hadoop Streaming or Java MapReduce.

In [10]:
%%writefile mapper_count_reviews.py
#!/usr/bin/env python3
import sys, json

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    try:
        obj = json.loads(line)
    except:
        continue
    asin = obj.get("asin")
    if asin:
        print(f"{asin}\t1")

Writing mapper_count_reviews.py


In [11]:
%%writefile reducer_count_reviews.py
#!/usr/bin/env python3
import sys

current = None
count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    asin, v = line.split("\t")
    v = int(v)

    if asin != current:
        if current is not None:
            print(f"{current}\t{count}")
        current = asin
        count = v
    else:
        count += v

if current is not None:
    print(f"{current}\t{count}")

Writing reducer_count_reviews.py


In [14]:
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -file mapper_count_reviews.py \
  -file reducer_count_reviews.py \
  -mapper mapper_count_reviews.py \
  -reducer reducer_count_reviews.py \
  -input movies.json \
  -output output_task1_reviews_count

2025-12-07 12:42:24,385 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_count_reviews.py, reducer_count_reviews.py] [] /tmp/streamjob9697143252843195585.jar tmpDir=null
2025-12-07 12:42:25,501 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2025-12-07 12:42:25,785 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2025-12-07 12:42:25,785 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2025-12-07 12:42:25,838 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2025-12-07 12:42:26,308 INFO mapred.FileInputFormat: Total input files to process : 1
2025-12-07 12:42:26,356 INFO mapreduce.JobSubmitter: number of splits:24
2025-12-07 12:42:26,858 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local399236203_0001
2025-12-07 12:42:26,859 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-12-07 12:42:27,460 INFO 

<h2>Task 2 — Compute Average Rating per Movie</h2>

<p>
In this task, you will compute the <b>average rating</b> for each movie.
</p>

<p>The mapper should:</p>
<ul>
  <li>Extract <code>asin</code> and <code>overall</code> (rating)</li>
  <li>Emit <code>(asin, rating)</code></li>
</ul>

<p>The reducer should:</p>
<ul>
  <li>Sum all ratings for each movie</li>
  <li>Count how many ratings were received</li>
  <li>Compute and output the average rating</li>
</ul>

<p>
Use a MapReduce job to generate a list of movies with their average ratings.
</p>

In [ ]:
# Write your Mapper and Reducer code for Task 2 here.
# You may use Python Hadoop Streaming or Java MapReduce.

In [15]:
%%writefile mapper_avg_rating.py
#!/usr/bin/env python3
import sys
import json

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    try:
        obj = json.loads(line)
    except:
        continue

    asin = obj.get("asin")
    overall = obj.get("overall")

    # convertir rating
    try:
        rating = float(overall)
    except:
        rating = None

    if asin and rating is not None:
        print(f"{asin}\t{rating}")

Writing mapper_avg_rating.py


In [16]:
%%writefile reducer_avg_rating.py
#!/usr/bin/env python3
import sys

current_asin = None
sum_r = 0.0
count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    try:
        asin, val = line.split("\t")
        rating = float(val)
    except:
        continue

    # nouvelle clé rencontrée
    if asin != current_asin:
        if current_asin is not None:
            avg = sum_r / count if count > 0 else 0
            print(f"{current_asin}\t{avg:.3f}")
        current_asin = asin
        sum_r = rating
        count = 1
    else:
        sum_r += rating
        count += 1

# dernière clé
if current_asin is not None:
    avg = sum_r / count if count > 0 else 0
    print(f"{current_asin}\t{avg:.3f}")


Writing reducer_avg_rating.py


In [17]:
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -file mapper_avg_rating.py \
  -file reducer_avg_rating.py \
  -mapper mapper_avg_rating.py \
  -reducer reducer_avg_rating.py \
  -input /content/movies.json \
  -output output_task2_avg_rating

2025-12-07 12:44:29,914 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_avg_rating.py, reducer_avg_rating.py] [] /tmp/streamjob12131315526237295506.jar tmpDir=null
2025-12-07 12:44:31,122 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2025-12-07 12:44:31,366 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2025-12-07 12:44:31,366 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2025-12-07 12:44:31,398 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2025-12-07 12:44:31,706 INFO mapred.FileInputFormat: Total input files to process : 1
2025-12-07 12:44:31,737 INFO mapreduce.JobSubmitter: number of splits:24
2025-12-07 12:44:32,074 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local2130198662_0001
2025-12-07 12:44:32,075 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-12-07 12:44:32,520 INFO mapr

<h2>Task 3 — Extract Frequent Keywords from Reviews</h2>

<p>
Now you will perform text analysis on the <code>reviewText</code> field.
Your task is to extract meaningful keywords for each movie.
</p>

<p>The mapper should:</p>
<ul>
  <li>Clean and tokenize the text</li>
  <li>Remove punctuation and stopwords</li>
  <li>Emit <code>(asin:word, 1)</code> for each keyword</li>
</ul>

<p>The reducer should:</p>
<ul>
  <li>Sum the counts for each <code>(asin, word)</code> pair</li>
  <li>Output the total frequency of each keyword per movie</li>
</ul>

<p>
This task combines text preprocessing with distributed computation.
</p>

In [ ]:
# Write your Mapper and Reducer code for Task 3 here.

In [18]:
%%writefile mapper_keywords.py
#!/usr/bin/env python3
import sys
import json
import re

# stopwords simples (tu peux étendre)
STOPWORDS = set([
    "the","and","a","an","of","to","is","in","it","this","that","for","on","with",
    "as","was","but","are","they","i","you","he","she","we","have","has","had",
    "not","movie","film","movies","films","one","two","use","using","so","if","at",
    "from","by","be","or","about","who","what","when","where","how"
])

WORD_RE = re.compile(r"[a-z0-9']+")

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    try:
        obj = json.loads(line)
    except:
        continue

    asin = obj.get("asin")
    text = obj.get("reviewText") or ""

    if not asin or not text:
        continue

    # nettoyage
    text = text.lower()
    tokens = WORD_RE.findall(text)

    for token in tokens:
        if token in STOPWORDS:
            continue
        if len(token) < 2:     # enlever "a", "i", etc.
            continue
        print(f"{asin}\t{token}\t1")

Writing mapper_keywords.py


In [19]:
%%writefile reducer_keywords.py
#!/usr/bin/env python3
import sys

current_key = None  # (asin, word)
count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    try:
        asin, word, v = line.split("\t")
        v = int(v)
    except:
        continue

    key = (asin, word)

    if key != current_key:
        if current_key is not None:
            a, w = current_key
            print(f"{a}\t{w}\t{count}")
        current_key = key
        count = v
    else:
        count += v

# flush final
if current_key is not None:
    a, w = current_key
    print(f"{a}\t{w}\t{count}")

Writing reducer_keywords.py


In [20]:
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -file mapper_keywords.py \
  -file reducer_keywords.py \
  -mapper mapper_keywords.py \
  -reducer reducer_keywords.py \
  -input /content/movies.json \
  -output output_task3_keywords_counts

2025-12-07 12:46:43,270 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_keywords.py, reducer_keywords.py] [] /tmp/streamjob570681366203199083.jar tmpDir=null
2025-12-07 12:46:44,315 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2025-12-07 12:46:44,519 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2025-12-07 12:46:44,519 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2025-12-07 12:46:44,557 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2025-12-07 12:46:44,879 INFO mapred.FileInputFormat: Total input files to process : 1
2025-12-07 12:46:44,910 INFO mapreduce.JobSubmitter: number of splits:24
2025-12-07 12:46:45,325 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1117225279_0001
2025-12-07 12:46:45,325 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-12-07 12:46:45,838 INFO mapred.Loc

<h2>Task 4 — Join Ratings with Top Keywords</h2>

<p>
For this task, you will combine the results of Task 2 (average ratings) and
Task 3 (keyword frequencies) using a <b>reduce-side join</b>.
</p>

<p>
You will provide two inputs to your MapReduce job:
</p>

<ul>
  <li><b>Ratings file</b> with <code>(asin, average_rating)</code></li>
  <li><b>Keywords file</b> with <code>(asin, keyword, count)</code></li>
</ul>

<p>Each mapper should tag its data:</p>

<ul>
  <li><code>("R", rating)</code> for ratings</li>
  <li><code>("K", keyword:count)</code> for keywords</li>
</ul>

<p>
The reducer will receive all entries for a given movie and combine them to
produce an output containing:
</p>

<ul>
  <li>The movie identifier (<code>asin</code>)</li>
  <li>Its average rating</li>
  <li>Its most frequent keywords</li>
</ul>

In [ ]:
# Write your Mapper and Reducer code for Task 4 here.

In [21]:
%%writefile mapper_join_ratings.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) < 2:
        continue

    asin = parts[0]
    avg = parts[1]

    print(f"{asin}\tR\t{avg}")

Writing mapper_join_ratings.py


In [22]:
%%writefile mapper_join_keywords.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) != 3:
        continue

    asin, word, count = parts

    print(f"{asin}\tK\t{word}:{count}")

Writing mapper_join_keywords.py


In [23]:
%%writefile reducer_join.py
#!/usr/bin/env python3
import sys

current_asin = None
rating = None
keywords = []   # list of (word, count)

def emit(asin, rating, keywords, top_n=10):
    if asin is None:
        return

    # Trier les mots par fréquence
    sorted_kw = sorted(keywords, key=lambda x: -x[1])
    top = sorted_kw[:top_n]

    # format
    top_str = ",".join([f"{w}({c})" for w, c in top])

    if rating is None:
        rstr = "NA"
    else:
        rstr = f"{rating:.3f}"

    print(f"{asin}\t{rstr}\t{top_str}")

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t", 2)
    if len(parts) < 3:
        continue

    asin, tag, payload = parts

    if asin != current_asin:
        if current_asin is not None:
            emit(current_asin, rating, keywords)
        current_asin = asin
        rating = None
        keywords = []

    if tag == "R":
        try:
            rating = float(payload)
        except:
            rating = None

    elif tag == "K":
        try:
            word, cnt = payload.rsplit(":", 1)
            keywords.append((word, int(cnt)))
        except:
            continue

# Dernière clé
emit(current_asin, rating, keywords)

Writing reducer_join.py


In [24]:
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -file mapper_join_ratings.py \
  -file mapper_join_keywords.py \
  -file reducer_join.py \
  -mapper mapper_join_ratings.py \
  -mapper mapper_join_keywords.py \
  -reducer reducer_join.py \
  -input output_task2_avg_rating \
  -input output_task3_keywords_counts \
  -output output_task4_join

2025-12-07 12:56:34,099 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_join_ratings.py, mapper_join_keywords.py, reducer_join.py] [] /tmp/streamjob6128341935343363853.jar tmpDir=null
2025-12-07 12:56:35,380 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2025-12-07 12:56:35,712 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2025-12-07 12:56:35,712 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2025-12-07 12:56:35,746 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2025-12-07 12:56:36,203 INFO mapred.FileInputFormat: Total input files to process : 2
2025-12-07 12:56:36,252 INFO mapreduce.JobSubmitter: number of splits:36
2025-12-07 12:56:36,791 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1959767363_0001
2025-12-07 12:56:36,791 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-12-07 12